In [2]:
import os
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv()

True

In [3]:
from langchain_neo4j import Neo4jGraph

# Neo4j Desktop 연결 설정
graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
    database=os.getenv("NEO4J_DATABASE"),
    enhanced_schema=True
)

In [4]:
# 테스트 쿼리 실행 
cypher_query = """
CREATE (n:Test {name: "Hello Neo4J DB"}) 
RETURN n
"""

graph.query(cypher_query)

[{'n': {'name': 'Hello Neo4J DB'}}]

## 2. 지식그래프 스키마 설계

* **주요 엔티티 (노드)**:

   1. **Document (문서)**: 
      - `id`: STRING - 문서의 고유 식별자 (예: '10k_data/tsla-20241231-gen.pdf')
      - `source`: STRING - 문서의 출처 (예: '10k_data/tsla-20241231-gen.pdf')

   2. **Section (섹션)**:
      - `name`: STRING - 섹션 이름 (예: "Business")
      - `document_id`: STRING - 소속 문서 ID (예: '10k_data/tsla-20241231-gen.pdf')

   3. **Chunk (청크)**:
      - `document_id`: STRING - 소속 문서 ID (예: '10k_data/tsla-20241231-gen.pdf')
      - `chunk_id`: STRING - 청크 고유 ID (예: "71ffd0db-fe12-45cb-8f33-81d89982ad60")
      - `section_start_page`: INTEGER - 섹션 시작 페이지 (범위: 5-98)
      - `section_name`: STRING - 소속 섹션 이름 (예: "Business")
      - `parent_id`: STRING - 부모 요소 ID (예: "736e213533bb2f95baf5ac36cd9fb0be")
      - `element_id`: STRING - 요소 ID (예: "bccf193d025e02740a6628e721ac2f73")
      - `content`: STRING - 실제 텍스트 내용 (예: "ITEM 1. BUSINESS  Overview  We design, develop, ma")
      - `order`: INTEGER - 순서 (범위: 1-112)
      - `embedding`: VECTOR - 벡터 임베딩 (차원: 1536)

* **관계**:

   1. **HAS_SECTION**: 문서가 섹션을 포함하는 관계
      - Document → Section

   2. **CONTAINS**: 섹션이 청크를 포함하는 관계
      - Section → Chunk

   3. **NEXT**: 청크 간의 순서 관계
      - Chunk → Chunk

* **제약조건**:

   1. Document의 id는 고유해야 함
   2. Section은 (name, document_id) 조합으로 고유하게 식별
   3. Chunk의 chunk_id는 고유해야 함

In [6]:
# Document 노드 레이블 및 속성 정의 (제약조건 설정)

cypher_query="""
CREATE CONSTRAINT IF NOT EXISTS // 제약조건 생성
FOR (d:Document) // Document 레이블을 가진 노드에 대해
REQUIRE d.id is UNIQUE; // id 속성이 유일해야 함
"""

graph.query(cypher_query)

[]

In [7]:
# Section 노드 레이블 및 속성 정의 (제약조건 설정)

cypher_query = """
CREATE CONSTRAINT IF NOT EXISTS // 제약조건 생성
FOR(s:Section) // Section 레이블을 가진 노드에 대해
REQUIRE (s.name, s.document_id) IS NODE KEY; // name과 document_id 속성이 유일해야함 (복합키)
"""

graph.query(cypher_query)

[]

In [8]:
# Chunk 노드 레이블 및 속성 정의 (제약조건 설정)

cypher_query = """
CREATE CONSTRAINT IF NOT EXISTS   // 제약조건 생성
FOR (c:Chunk)  // Chunk 레이블을 가진 노드에 대해
REQUIRE c.chunk_id IS UNIQUE;  // chunk_id 속성이 유일해야 함
"""

graph.query(cypher_query)

[]

In [ ]:
# 벡터 인덱스 생성
# 저장되어있는 Chunk를 기준으로 검색할 수 있게 indexing 해줌

cypher_query ="""
CREATE VECTOR INDEX chunk_content_index IF NOT EXISTS
FOR (c:Chunk)
ON (c.embedding)
OPTIONS {
    indexConfig : {
        `vector.dimensions`: 3072,
        `vector.similarity_function` : 'cosine'
    }
}

"""

graph.query (cypher_query)

[]